In [663]:
import requests
import re
from bs4 import BeautifulSoup
import pandas as pd
import os

manager_df = pd.DataFrame()
iteration_number = 21
reader = pd.read_csv("Football-Data/Team_List.csv")
print(reader['Squad'])

0               Liverpool FC
1                 Arsenal FC
2                 Chelsea FC
3     Brighton & Hove Albion
4            Manchester City
               ...          
91                Angers SCO
92          AS Saint-Étienne
93               Le Havre AC
94                 FC Nantes
95           Montpellier HSC
Name: Squad, Length: 96, dtype: object


In [664]:
Names_List = []
Ages_List = []
Nationality_List = []
Team_List = []
Contract_Expiry_List = []
Page_Link = []
Name_ID = []
Manager_ID = []
Manager_Profile = []

In [665]:
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.106 Safari/537.36'}
page = 'https://www.transfermarkt.us/carlo-ancelotti/profil/trainer/523'
pageTree = requests.get(page, headers = headers)
pageSoup = BeautifulSoup(pageTree.content, 'html.parser')

In [666]:
Name_Scraper = pageSoup.find("h1", {"class": "data-header__headline-wrapper"})
Name_Details = pageSoup.find("div", {"class": "data-header__details"}).find_all("span", {"class": "data-header__content"})
Team_Details = pageSoup.find("div", {"class": "data-header__club-info"}).find_all("span", {"class": ["data-header__club", "data-header__content"]})
Manager_Profile_Search = pageSoup.find("div", {"class": "modal-trigger"}).find_all("img", {"class": "data-header__profile-image"})
for i in range(len(Name_Details)): 
    Name_Details[i] = BeautifulSoup(str(Name_Details[i]), 'html.parser').text.strip()
for j in range(len(Team_Details)): 
    Team_Details[j] = BeautifulSoup(str(Team_Details[j]), 'html.parser').text.strip()

soup = BeautifulSoup(str(Manager_Profile_Search[0]), "html.parser")
img_tag = soup.find('img', class_='data-header__profile-image').get('src')

print(Name_Details)
print(Team_Details)

['Jun 10, 1959 (65)', 'Reggiolo', 'Italy', 'UEFA Pro Licence', '2.35 Years', '4-3-3 Attacking']
['Real Madrid', 'Jul 1, 2021', 'Jun 30, 2026']


In [667]:
Names_List.append([BeautifulSoup(str(Name_Scraper), 'html.parser').text.strip()][0])
Ages_List.append(re.search(r"\((\d+)\)", Name_Details[0]).group(1))
Nationality_List.append(Name_Details[2])
Team_List.append(reader['Squad'][iteration_number])
if len(Team_Details) == 2:
    Contract_Expiry_List.append(Contract_Expiry_List[len(Contract_Expiry_List)-1])
else:
    Contract_Expiry_List.append(Team_Details[2])
Page_Link.append(page)
Name_ID.append(str(page).split('/')[3])
Manager_ID.append(str(page).split('/')[6])
Manager_Profile.append(img_tag)

In [668]:
manager_df = pd.DataFrame({"Names":Names_List,
                     "Age":Ages_List,
                     "Team": Team_List,
                     "Nationality":Nationality_List,
                     "Contract Expiration":Contract_Expiry_List,
                     "Page Link": Page_Link,
                     "Name ID": Name_ID,
                     "Manager ID":Manager_ID,
                     "Manager Profile":Manager_Profile})

print(manager_df)

             Names Age         Team Nationality Contract Expiration  \
0  Carlo Ancelotti  65  Real Madrid       Italy        Jun 30, 2026   

                                           Page Link          Name ID  \
0  https://www.transfermarkt.us/carlo-ancelotti/p...  carlo-ancelotti   

  Manager ID                                    Manager Profile  
0        523  https://img.a.transfermarkt.technology/portrai...  


In [ ]:
downloads_folder = os.path.expanduser("~/Downloads")
file_path = os.path.join(downloads_folder, "Manager_List2.csv")
manager_df.to_csv(file_path, index=False)